# Reconstruction Attack

**Goal:** Train a decoder that maps 128-dim CNN embeddings back to X-ray images.

**Architecture (from privacy_ml/models.py):**
```
Image (150x150x1)
  → Conv blocks
  → Flatten (36,992)
  → Dense(128, relu)  ← this is the embedding sent to cloud
  → Dense(1, sigmoid) ← cloud classifier head
```

**Attacker has:** 128-dim embedding vectors

**Attacker goal:** 128-dim → 150x150 X-ray image

**Metrics (slide 29):** MSE, PSNR, SSIM

In [ ]:
# ================================
# 1. Mount Google Drive & install dependencies
# ================================
!pip install -q scikit-image

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ================================
# 2. Clone repo to use privacy_ml package
# ================================
import os, sys

if not os.path.exists('/content/Privacy_Preserving_ML'):
    !git clone https://github.com/Erayisci/Privacy_Preserving_ML.git

sys.path.insert(0, '/content/Privacy_Preserving_ML')

In [ ]:
# ================================
# 3. Imports
# ================================
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path
from tensorflow.keras import layers, Model, Input
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

from privacy_ml.models import (
    build_encoder, build_head, build_end_to_end,
    compile_for_binary_classification,
    DEFAULT_IMG_SIZE, DEFAULT_CHANNELS, EMBEDDING_DIM, DEFAULT_DROPOUT_RATE
)
from privacy_ml.data import load_kaggle_pool, split_pool_indices

IMG_SIZE = DEFAULT_IMG_SIZE   # 150
EMBED_DIM = EMBEDDING_DIM     # 128
SEED = 42

In [ ]:
# ================================
# 4. Load dataset from Google Drive
# ================================
# "Benimle paylaşılanlar" altındaki chest_xray klasörü
base_dir = Path('/content/drive/MyDrive/chest_xray')

X, y = load_kaggle_pool(base_dir, img_size=IMG_SIZE)
print(f"Total images: {X.shape}  Labels: {y.shape}")

splits = split_pool_indices(y, seed=SEED)

X_train = X[splits.victim_members]
y_train = y[splits.victim_members]

X_test  = X[splits.victim_nonmembers]
y_test  = y[splits.victim_nonmembers]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# ================================
# 5. Laplace noise function (same as Untitled3.ipynb)
# ================================
def add_laplace_noise(images: np.ndarray, epsilon: float = 1.0) -> np.ndarray:
    scale = 1.0 / epsilon
    noise = np.random.laplace(0.0, scale, size=images.shape)
    return np.clip(images + noise, 0.0, 1.0)

In [ ]:
# ================================
# 6. Build & train CNN models using backbone
# ================================
def make_and_train(X_tr, y_tr, X_val, y_val, name, epochs=10):
    encoder = build_encoder(IMG_SIZE, DEFAULT_CHANNELS, EMBED_DIM, name=f"{name}_enc")
    head    = build_head(EMBED_DIM, DEFAULT_DROPOUT_RATE, name=f"{name}_head")
    model   = build_end_to_end(encoder, head, name=name)
    compile_for_binary_classification(model, learning_rate=1e-3)
    model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
              epochs=epochs, batch_size=32, verbose=1)
    return model, encoder

# Validation slice
X_val = X[splits.shadow_pool[:200]]
y_val = y[splits.shadow_pool[:200]]

# Baseline model (no privacy)
model_normal, encoder_normal = make_and_train(
    X_train, y_train, X_val, y_val, name='baseline'
)

# Laplace DP model
X_train_noisy = add_laplace_noise(X_train, epsilon=1.0)
model_laplace, encoder_laplace = make_and_train(
    X_train_noisy, y_train, X_val, y_val, name='laplace'
)

print("Baseline test acc:", model_normal.evaluate(X_test, y_test, verbose=0)[1])
print("Laplace  test acc:", model_laplace.evaluate(add_laplace_noise(X_test), y_test, verbose=0)[1])

## Reconstruction Attack

Attacker obtains 128-dim embeddings and trains a decoder to recover 150x150 images.

In [ ]:
# ================================
# 7. Extract 128-dim embeddings
# ================================
Z_normal  = encoder_normal.predict(X_test,  verbose=0)  # (1000, 128)

X_test_noisy = add_laplace_noise(X_test, epsilon=1.0)
Z_laplace = encoder_laplace.predict(X_test_noisy, verbose=0)  # (1000, 128)

print("Embedding shape (normal) :", Z_normal.shape)
print("Embedding shape (laplace):", Z_laplace.shape)

In [ ]:
# ================================
# 8. Build Decoder
#    128-dim → 150x150x1
# ================================
def build_decoder(embed_dim: int = EMBED_DIM, img_size: int = IMG_SIZE) -> Model:
    inp = Input(shape=(embed_dim,))

    # Expand to spatial feature map
    x = layers.Dense(7 * 7 * 128, activation='relu')(inp)
    x = layers.Reshape((7, 7, 128))(x)

    # Upsample:  7 → 14 → 28 → 56 → 112
    for filters in [128, 64, 32, 16]:
        x = layers.Conv2DTranspose(filters, (3,3), strides=2, padding='same', activation='relu')(x)

    # Resize 112 → 150
    x = layers.Resizing(img_size, img_size)(x)

    # Final pixel output
    x = layers.Conv2D(1, (3,3), padding='same', activation='sigmoid')(x)

    return Model(inp, x, name='decoder')

build_decoder().summary()

In [ ]:
# ================================
# 9. Train decoder — Baseline (no privacy)
# ================================
decoder_normal = build_decoder()
decoder_normal.compile(optimizer='adam', loss='mse')
decoder_normal.fit(
    Z_normal, X_test,
    epochs=50, batch_size=32,
    validation_split=0.1, verbose=1
)

In [ ]:
# ================================
# 10. Train decoder — Differential Privacy (Laplace)
# ================================
decoder_laplace = build_decoder()
decoder_laplace.compile(optimizer='adam', loss='mse')
decoder_laplace.fit(
    Z_laplace, X_test,
    epochs=50, batch_size=32,
    validation_split=0.1, verbose=1
)

In [ ]:
# ================================
# 11. Compute Metrics
# ================================
recon_normal  = decoder_normal.predict(Z_normal)
recon_laplace = decoder_laplace.predict(Z_laplace)

def compute_metrics(originals, reconstructed, label):
    mse_vals, psnr_vals, ssim_vals = [], [], []
    for orig, recon in zip(originals, reconstructed):
        o, r = orig[:,:,0], recon[:,:,0]
        mse_vals.append(np.mean((o - r)**2))
        psnr_vals.append(psnr(o, r, data_range=1.0))
        ssim_vals.append(ssim(o, r, data_range=1.0))
    print(f"\n--- {label} ---")
    print(f"MSE  (↑ better privacy): {np.mean(mse_vals):.4f}")
    print(f"PSNR (↓ better privacy): {np.mean(psnr_vals):.2f} dB")
    print(f"SSIM (↓ better privacy): {np.mean(ssim_vals):.4f}")
    return np.mean(mse_vals), np.mean(psnr_vals), np.mean(ssim_vals)

mse_n, psnr_n, ssim_n = compute_metrics(X_test, recon_normal,  "Baseline (no privacy)")
mse_l, psnr_l, ssim_l = compute_metrics(X_test, recon_laplace, "Differential Privacy (Laplace)")

In [ ]:
# ================================
# 12. Summary Table
# ================================
print("\n" + "="*52)
print(f"{'Metric':<22} {'Baseline':>12} {'DP (Laplace)':>14}")
print("="*52)
print(f"{'MSE (↑=safer)':<22} {mse_n:>12.4f} {mse_l:>14.4f}")
print(f"{'PSNR dB (↓=safer)':<22} {psnr_n:>12.2f} {psnr_l:>14.2f}")
print(f"{'SSIM (↓=safer)':<22} {ssim_n:>12.4f} {ssim_l:>14.4f}")
print("="*52)
print("Expected (slide 29): High MSE, Low PSNR, SSIM ≈ 0")

In [ ]:
# ================================
# 13. Visual Comparison
# ================================
n = 4
idx = np.random.choice(len(X_test), n, replace=False)

fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
fig.suptitle('Reconstruction Attack: Baseline vs DP (Laplace)', fontsize=14)

for col, title in enumerate(['Original', 'Reconstructed (No Privacy)', 'Reconstructed (DP)']):
    axes[0, col].set_title(title, fontweight='bold')

for row, i in enumerate(idx):
    axes[row, 0].imshow(X_test[i,:,:,0],        cmap='gray', vmin=0, vmax=1)
    axes[row, 1].imshow(recon_normal[i,:,:,0],  cmap='gray', vmin=0, vmax=1)
    axes[row, 2].imshow(recon_laplace[i,:,:,0], cmap='gray', vmin=0, vmax=1)
    for col in range(3):
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('reconstruction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ================================
# 14. Bar Chart
# ================================
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
labels = ['Baseline', 'DP (Laplace)']
colors = ['#e74c3c', '#2ecc71']

axes[0].bar(labels, [mse_n,  mse_l],  color=colors)
axes[0].set_title('MSE (higher = better privacy)')

axes[1].bar(labels, [psnr_n, psnr_l], color=colors)
axes[1].set_title('PSNR dB (lower = better privacy)')

axes[2].bar(labels, [ssim_n, ssim_l], color=colors)
axes[2].set_title('SSIM (lower = better privacy)')

plt.suptitle('Reconstruction Attack Metrics', fontsize=13)
plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()